# 06 - Volume Prediction

Volume leads price. Detect surges.


## Setup

In [1]:
# --- setup --------------------------------------------------------
import sys, json
sys.path.insert(0, r"D:\\Projects\\done\\eth_bot\\eth_bot_v2")
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (13, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
np.random.seed(42)

SYMBOL = "ETHUSDT"
OUT = Path(r"D:\\Projects\\done\\eth_bot\\eth_bot_v2") / "reports" / "forecasts"
OUT.mkdir(parents=True, exist_ok=True)
DB = Path(r"D:\\Projects\\done\\eth_bot\\eth_bot_v2") / "database" / "market.db"

def sql(q, **p):
    with sqlite3.connect(DB) as c:
        return pd.read_sql_query(q, c, params=p)

from database.config.api.binance import get_klines

def fetch(symbol, interval, limit=500):
    raw = get_klines(symbol, interval=interval, limit=limit)
    df = pd.DataFrame(raw, columns=["open_time","open","high","low","close",
        "volume","close_time","quote_volume","trades","taker_base","taker_quote","ignore"])
    df["date"] = pd.to_datetime(df["open_time"], unit="ms", utc=True).dt.tz_convert(None)
    for c in ("open","high","low","close","volume"):
        df[c] = df[c].astype(float)
    return df[["date","open","high","low","close","volume"]].set_index("date").iloc[:-1]

def action_block(name, action, reason, price=None, entry=None, stop=None,
                 target=None, conf=None, extra=None):
    print("=" * 62)
    print(f"  {name}")
    print("=" * 62)
    print(f"  time   : {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M')} UTC")
    if price is not None: print(f"  price  : {price:,.2f}")
    if conf  is not None: print(f"  conf   : {conf:.2f}")
    print(f"  ACTION : {action}")
    print(f"  REASON : {reason}")
    if entry is not None:
        print(f"  entry  : {entry:,.2f}   stop: {stop:,.2f}   target: {target:,.2f}")
    if extra:
        for k, v in extra.items():
            print(f"  {k:<6} : {v}")
    print("=" * 62)
    return {"action": action, "reason": reason, "price": price,
            "entry": entry, "stop": stop, "target": target, "conf": conf}

## 1. Load volume history

In [2]:
v = sql("SELECT open_time, close, volume FROM ohlcv WHERE symbol='ETHUSDT' AND timeframe='1d' ORDER BY open_time")
v["date"] = pd.to_datetime(v.open_time, unit="ms")
v = v.set_index("date").drop(columns="open_time")
print("n=", len(v))

n= 2189


## 2. Volume z-score

In [3]:
v["vol_z"] = (v["volume"] - v["volume"].rolling(50).mean()) / v["volume"].rolling(50).std()
print("current volume z-score:", round(v["vol_z"].iloc[-1], 2))
print("last 5 z-scores:")
print(v["vol_z"].tail(5))

current volume z-score: 0.68
last 5 z-scores:
date
2026-09-28   -0.057092
2026-09-29    0.074436
2026-09-30   -0.199341
2026-10-01   -0.263393
2026-10-02    0.676568
Name: vol_z, dtype: float64


## 3. Price effect of high-volume days

In [4]:
v["ret"] = v["close"].pct_change()
high_vol = v[v["vol_z"] > 1.5]["ret"].dropna()
low_vol  = v[v["vol_z"] < -1.0]["ret"].dropna()
print(f"high-vol mean return : {high_vol.mean()*100:+.3f}%  (n={len(high_vol)})")
print(f"low-vol mean return  : {low_vol.mean()*100:+.3f}%  (n={len(low_vol)})")

high-vol mean return : -0.482%  (n=167)
low-vol mean return  : +0.185%  (n=307)


## 4. Action

In [5]:
z = float(v["vol_z"].iloc[-1])
ret_today = float(v["ret"].iloc[-1])
if z > 1.5 and ret_today > 0:   action, reason = "LONG",  "high volume + up day — continuation"
elif z > 1.5 and ret_today < 0: action, reason = "SHORT", "high volume + down day — capitulation"
else:                            action, reason = "FLAT",  "normal volume — no signal"
action_block("VOLUME", action, reason,
    price=float(v["close"].iloc[-1]), extra={"vol_z": f"{z:.2f}"})

  VOLUME
  time   : 2026-10-04 10:42 UTC
  price  : 2,668.59
  ACTION : FLAT
  REASON : normal volume — no signal
  vol_z  : 0.68


{'action': 'FLAT',
 'reason': 'normal volume — no signal',
 'price': 2668.59,
 'entry': None,
 'stop': None,
 'target': None,
 'conf': None}